In [ ]:
#Import Libraries
import os
import subprocess
import shlex
import socket
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.memory import MemorySaver  # replaces ConversationBufferMemory
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent


import os
os.environ["OPENAI_API_KEY"] = "Put your Open API Key here"


In [16]:
def calculate_network_range(ip_addr, cidr):
    try:
        cidr_int = int(cidr)
        ip_parts = [int(x) for x in ip_addr.split('.')]
        host_bits = 32 - cidr_int
        mask = (0xFFFFFFFF << host_bits) & 0xFFFFFFFF
        network_addr = []
        for i in range(4):
            network_addr.append(ip_parts[i] & ((mask >> (24 - i * 8)) & 0xFF))
        return f"{'.'.join(map(str, network_addr))}/{cidr}"
    except:
        return None


def netmask_to_cidr(netmask):
    try:
        parts = netmask.split('.')
        binary = ''.join([bin(int(part))[2:].zfill(8) for part in parts])
        return binary.count('1')
    except:
        return 24

In [ ]:
#Define tools
@tool
def get_local_network_info() -> str:
    """Get local network information including IP addresses, subnet masks, and network ranges for scanning."""
    try:
        hostname = socket.gethostname()

        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        try:
            s.connect(("8.8.8.8", 80))
            primary_ip = s.getsockname()[0]
            s.close()
        except:
            primary_ip = "Unable to determine"

        try:
            all_ips = socket.gethostbyname_ex(hostname)[2]
        except:
            all_ips = []

        interface_info = []
        scan_ranges = []

        try:
            result = subprocess.run(["ip", "addr", "show"], capture_output=True, text=True, timeout=5)
            if result.returncode == 0:
                current_interface = None
                for line in result.stdout.split('\n'):
                    if re.match(r'^\d+:', line):
                        current_interface = line.split(':')[1].strip()
                    elif 'inet ' in line and current_interface:
                        ip_match = re.search(r'inet (\d+\.\d+\.\d+\.\d+)/(\d+)', line)
                        if ip_match and not ip_match.group(1).startswith('127.'):
                            ip_addr = ip_match.group(1)
                            cidr = ip_match.group(2)
                            interface_info.append(f"{current_interface}: {ip_addr}/{cidr}")
                            network_range = calculate_network_range(ip_addr, cidr)
                            if network_range and network_range not in scan_ranges:
                                scan_ranges.append(network_range)
        except:
            try:
                result = subprocess.run(["ifconfig"], capture_output=True, text=True, timeout=5)
                if result.returncode == 0:
                    current_interface = None
                    for line in result.stdout.split('\n'):
                        if line and not line.startswith(' ') and not line.startswith('\t'):
                            current_interface = line.split()[0].rstrip(':')
                        if 'inet ' in line and current_interface:
                            ip_match = re.search(r'inet (\d+\.\d+\.\d+\.\d+)', line)
                            if ip_match and not ip_match.group(1).startswith('127.'):
                                ip_addr = ip_match.group(1)
                                netmask_match = re.search(r'netmask (\d+\.\d+\.\d+\.\d+)', line)
                                if netmask_match:
                                    cidr = netmask_to_cidr(netmask_match.group(1))
                                    interface_info.append(f"{current_interface}: {ip_addr}/{cidr}")
                                    network_range = calculate_network_range(ip_addr, str(cidr))
                                    if network_range and network_range not in scan_ranges:
                                        scan_ranges.append(network_range)
                                else:
                                    interface_info.append(f"{current_interface}: {ip_addr}")
            except:
                pass

        response = f"Hostname: {hostname}\nPrimary IP: {primary_ip}\n"
        if all_ips:
            response += f"All IPs for hostname: {', '.join(all_ips)}\n"
        if interface_info:
            response += "\nNetwork interfaces:\n" + "".join(f"  {i}\n" for i in interface_info)
        if scan_ranges:
            response += "\nNetwork ranges (for scanning):\n" + "".join(f"  {r}\n" for r in scan_ranges)
        else:
            if primary_ip != "Unable to determine":
                octets = primary_ip.split('.')
                if len(octets) == 4:
                    response += f"\nSuggested scan range (assuming /24): {octets[0]}.{octets[1]}.{octets[2]}.0/24\n"

        response += "\nNote: Use the network ranges above with nmap_ping_scan to discover all devices."
        return response

    except Exception as ex:
        return f"Error getting network info: {type(ex).__name__}: {ex}"


In [18]:
@tool
def nmap_scan(target: str, options: str = "") -> str:
    """Run an Nmap scan on the specified target with optional parameters.
    
    Args:
        target: The target to scan (IP address, hostname, or IP range)
        options: Additional Nmap command line options (e.g., '-sS -p 80,443')
    """
    cmd_parts = ["nmap"]
    if options:
        cmd_parts.extend(shlex.split(options))
    cmd_parts.append(target)
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=300, check=False)
        if result.returncode != 0:
            return f"Error: Nmap returned non-zero exit code {result.returncode}\nStderr: {result.stderr}"
        return result.stdout
    except subprocess.TimeoutExpired:
        return "Error: Nmap scan timed out after 5 minutes"
    except FileNotFoundError:
        return "Error: nmap command not found. Please install nmap first."
    except Exception as ex:
        return f"Error: {type(ex).__name__}: {ex}"


@tool
def nmap_quick_scan(target: str) -> str:
    """Perform a quick TCP scan on the most common ports. Equivalent to: nmap -T4 -F target
    
    Args:
        target: The target to scan
    """
    return nmap_scan.invoke({"target": target, "options": "-T4 -F"})


@tool
def nmap_port_scan(target: str, ports: str) -> str:
    """Scan specific ports on the target.
    
    Args:
        target: The target to scan
        ports: Port specification (e.g., '80', '80,443', '1-1000')
    """
    return nmap_scan.invoke({"target": target, "options": f"-p {ports}"})


@tool
def nmap_service_detection(target: str, ports: str = "") -> str:
    """Perform service version detection on the target.
    
    Args:
        target: The target to scan
        ports: Optional port specification
    """
    options = "-sV"
    if ports:
        options += f" -p {ports}"
    return nmap_scan.invoke({"target": target, "options": options})


@tool
def nmap_os_detection(target: str) -> str:
    """Attempt to detect the operating system of the target. Requires root/sudo privileges.
    
    Args:
        target: The target to scan
    """
    return nmap_scan.invoke({"target": target, "options": "-O"})


@tool
def nmap_ping_scan(target: str) -> str:
    """Perform a ping scan to discover live hosts with no port scanning.
    
    Args:
        target: The target or range to scan (e.g., '192.168.1.0/24')
    """
    return nmap_scan.invoke({"target": target, "options": "-sn"})


@tool
def nmap_script_scan(target: str, script: str, ports: str = "") -> str:
    """Run a specific Nmap script against the target.
    
    Args:
        target: The target to scan
        script: The script name or category (e.g., 'http-title', 'vuln')
        ports: Optional port specification
    """
    options = f"--script {script}"
    if ports:
        options += f" -p {ports}"
    return nmap_scan.invoke({"target": target, "options": options})


In [19]:
#Agent setup 
tools = [
    get_local_network_info,
    nmap_scan,
    nmap_quick_scan,
    nmap_port_scan,
    nmap_service_detection,
    nmap_os_detection,
    nmap_ping_scan,
    nmap_script_scan,
]

llm = ChatOpenAI(model="gpt-4o", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a network security assistant. Use the available nmap tools to help the user scan and analyze networks."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
#Run
if __name__ == "__main__":
    result = agent_executor.invoke({"input": "Return all the local network info"})
    print(result["output"])